In [21]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score



In [27]:
np.random.seed(42)
n = 2000

X1 = np.random.normal(0, 1, n)
X2 = 2.0 * X1 + np.random.normal(0, 0.6, n)
X3 = -1.0 * X1 + 0.5 * X2 + np.random.normal(0, 0.6, n)

# downstream target
y = 3.0 * X1 - 1.5 * X2 + 2.0 * X3 + np.random.normal(0, 1.0, n)

df_complete = pd.DataFrame({
    "X1": X1,
    "X2": X2,
    "X3": X3,
    "y": y
})


In [28]:
df_missing = df_complete.copy()

missing_rate = 0.05
mask = np.random.rand(*df_missing.shape) < missing_rate

# keep target fully observed
mask[:, df_missing.columns.get_loc("y")] = False

df_missing[mask] = np.nan


In [29]:
mean_imputer = SimpleImputer(strategy="mean")

df_mean = pd.DataFrame(
    mean_imputer.fit_transform(df_missing),
    columns=df_missing.columns
)


In [30]:
def linear_impute_single_pass(df):
    df_imp = df.copy()
    predictor_imputer = SimpleImputer(strategy="mean")

    for col in df.columns:
        missing = df[col].isna()
        if missing.sum() == 0:
            continue

        X = df.drop(columns=[col])
        y_col = df[col]

        X_filled = predictor_imputer.fit_transform(X)

        X_train = X_filled[~missing]
        y_train = y_col[~missing]
        X_pred = X_filled[missing]

        model = LinearRegression()
        model.fit(X_train, y_train)

        df_imp.loc[missing, col] = model.predict(X_pred)

    return df_imp


df_linear = linear_impute_single_pass(df_missing)


In [31]:
mice_imputer = IterativeImputer(
    random_state=42,
    max_iter=10,
    sample_posterior=True
)

df_mice = pd.DataFrame(
    mice_imputer.fit_transform(df_missing),
    columns=df_missing.columns
)


In [32]:
def rmse_on_mask(imputed, true, mask):
    return np.sqrt(
        mean_squared_error(
            true.values[mask],
            imputed.values[mask]
        )
    )

rmse_mean = rmse_on_mask(df_mean, df_complete, mask)
rmse_linear = rmse_on_mask(df_linear, df_complete, mask)
rmse_mice = rmse_on_mask(df_mice, df_complete, mask)

print("RMSE Mean:  ", rmse_mean)
print("RMSE Linear:", rmse_linear)
print("RMSE MICE:  ", rmse_mice)


RMSE Mean:   1.3510541970394834
RMSE Linear: 0.4196994585150404
RMSE MICE:   0.6730921710749384


In [33]:
num_cols = ["X1", "X2", "X3"]

def dist_stats(df):
    return pd.DataFrame({
        "mean": df[num_cols].mean(),
        "var": df[num_cols].var(),
        "skew": df[num_cols].skew()
    })

stats_true = dist_stats(df_complete)
stats_mean = dist_stats(df_mean)
stats_linear = dist_stats(df_linear)
stats_mice = dist_stats(df_mice)

print("ABS shift — Mean imputation")
print((stats_mean - stats_true).abs())

print("\nABS shift — Linear imputation")
print((stats_linear - stats_true).abs())

print("\nABS shift — MICE imputation")
print((stats_mice - stats_true).abs())


ABS shift — Mean imputation
        mean       var      skew
X1  0.002459  0.033981  0.000697
X2  0.002905  0.216305  0.009275
X3  0.000519  0.023638  0.001348

ABS shift — Linear imputation
        mean       var      skew
X1  0.000194  0.000781  0.001693
X2  0.000821  0.019754  0.003610
X3  0.000510  0.012661  0.000661

ABS shift — MICE imputation
        mean       var      skew
X1  0.001054  0.005497  0.000595
X2  0.003186  0.021848  0.008876
X3  0.000469  0.001891  0.000602


In [34]:
def corr_shift(df_a, df_b):
    return np.linalg.norm(
        df_a[num_cols].corr().values - df_b[num_cols].corr().values
    )

print("Corr shift Mean:  ", corr_shift(df_mean, df_complete))
print("Corr shift Linear:", corr_shift(df_linear, df_complete))
print("Corr shift MICE:  ", corr_shift(df_mice, df_complete))


Corr shift Mean:   0.058758771873083704
Corr shift Linear: 0.005219522597949869
Corr shift MICE:   0.008719792670851927


In [35]:
def cv_r2(df):
    X = df[["X1", "X2", "X3"]]
    y = df["y"]
    model = LinearRegression()
    return cross_val_score(model, X, y, cv=5, scoring="r2").mean()

print("R² Mean:  ", cv_r2(df_mean))
print("R² Linear:", cv_r2(df_linear))
print("R² MICE:  ", cv_r2(df_mice))


R² Mean:   0.4194740626136383
R² Linear: 0.6208228900790074
R² MICE:   0.5928929842211982
